# Numerikus módszerek -- 3. előadás + gyakorlat
## Progonka módszer, LDU-felbontás, Cholesky-felbontás

Ez a notebook **részletes kidolgozott példákat** és **önálló feladatokat** tartalmaz a következő témákhoz:

1. Progonka módszer (rövidített GE tridiagonális rendszerekre)
2. LDU-felbontás (és LDL$^\top$ szimmetrikus esetben)
3. Cholesky-felbontás ($LL^\top$)
4. Önálló feladatok
5. Debuggolási feladatok

In [ ]:
import numpy as np
from scipy.linalg import lu, cho_factor, cho_solve
import time

np.set_printoptions(precision=4, suppress=True)

def print_matrix(M, label=""):
    """Mátrix szép kiírása."""
    if label:
        print(label)
    for i in range(M.shape[0]):
        row = "  ".join(f"{M[i,j]:8.4f}" for j in range(M.shape[1]))
        print(f"  [{row}]")
    print()

---
## 1. Progonka módszer (rövidített GE tridiagonális rendszerekre)

### Motiváció

A gyakorlatban megszokott, hogy **tridiagonális** (háromátlós) LER-t kell megoldanunk (pl. köbös spline-ok meghatározásánál). A speciális alakot felhasználva hatékonyabb algoritmust készíthetünk:

| | Általános GE | Progonka |
|---|---|---|
| **Tárolás** | $n^2$ | $3n - 2$ |
| **Műveletigény** | $\frac{2}{3}n^3 + \mathcal{O}(n^2)$ | $8n + \mathcal{O}(1)$ |

### Jelölések

Legyen $A = \text{tridiag}(\beta_{i-1}, \alpha_i, \gamma_i)$, azaz:

$$
A = \begin{bmatrix}
\alpha_1 & \gamma_1 & 0 & \cdots & 0 \\
\beta_1 & \alpha_2 & \gamma_2 & & 0 \\
0 & \ddots & \ddots & \ddots & \vdots \\
\vdots & & \beta_{n-2} & \alpha_{n-1} & \gamma_{n-1} \\
0 & & 0 & \beta_{n-1} & \alpha_n
\end{bmatrix}, \quad
x = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_{n-1} \\ x_n \end{bmatrix}, \quad
b = \begin{bmatrix} b_1 \\ b_2 \\ \vdots \\ b_{n-1} \\ b_n \end{bmatrix}
$$

### Az algoritmus levezetése

A GE a sávszélességet megtartja: tridiagonális esetben a GE végén kapott $U$ mátrix is csak két átlót tartalmaz. A visszahelyettesítés $i$. egyenlete:

$$a_{ii}^{(i-1)} x_i + a_{i,i+1}^{(i-1)} x_{i+1} = a_{i,n+1}^{(i-1)}$$

Ebből $x_i$-t kifejezve, új jelölésrendszerrel $x_i = f_i x_{i+1} + g_i$ alakú.

**Az 1. egyenletből** ($i = 1$):

$$\alpha_1 x_1 + \gamma_1 x_2 = b_1 \quad \Rightarrow \quad x_1 = -\frac{\gamma_1}{\alpha_1} x_2 + \frac{b_1}{\alpha_1}$$

Tehát $f_1 = -\frac{\gamma_1}{\alpha_1}$ és $g_1 = \frac{b_1}{\alpha_1}$.

**Az $i$-edik egyenletbe** ($i = 2, \ldots, n-1$) behelyettesítve $x_{i-1} = f_{i-1} x_i + g_{i-1}$:

$$\beta_{i-1}(f_{i-1}x_i + g_{i-1}) + \alpha_i x_i + \gamma_i x_{i+1} = b_i$$

$$(\alpha_i + \beta_{i-1} f_{i-1}) x_i + \gamma_i x_{i+1} = b_i - \beta_{i-1} g_{i-1}$$

Innen:
$$f_i = -\frac{\gamma_i}{\alpha_i + \beta_{i-1} f_{i-1}}, \qquad g_i = \frac{b_i - \beta_{i-1} g_{i-1}}{\alpha_i + \beta_{i-1} f_{i-1}}$$

**Az $n$-edik egyenletből** (nincs $x_{n+1}$):

$$x_n = \frac{b_n - \beta_{n-1} g_{n-1}}{\alpha_n + \beta_{n-1} f_{n-1}} =: g_n$$

### Algoritmus: progonka módszer

**1. lépés (előre):**

$$f_1 := -\frac{\gamma_1}{\alpha_1}, \qquad g_1 := \frac{b_1}{\alpha_1}$$

$$i = 2, \ldots, n-1: \quad f_i := -\frac{\gamma_i}{\alpha_i + \beta_{i-1} f_{i-1}}, \qquad g_i := \frac{b_i - \beta_{i-1} g_{i-1}}{\alpha_i + \beta_{i-1} f_{i-1}}$$

$$g_n := \frac{b_n - \beta_{n-1} g_{n-1}}{\alpha_n + \beta_{n-1} f_{n-1}}$$

**2. lépés (vissza):**

$$x_n := g_n$$
$$i = n-1, n-2, \ldots, 1: \quad x_i = f_i x_{i+1} + g_i$$

### Műveletigény

- **1. lépés (előre):** $f_1, g_1$: 2 művelet. A ciklusban: közös nevező 2 db, $f_i$ 1 db, $g_i$ 3 db → összesen $6(n-2)$ db. $g_n$: 5 db.
- **2. lépés (vissza):** $x_i = f_i x_{i+1} + g_i$ → $2(n-1)$ db.
- **Összesen:** $2 + 6(n-2) + 5 + 2(n-1) = 8n - 7 = 8n + \mathcal{O}(1)$ művelet.

### Kidolgozott példa

Oldjuk meg a következő tridiagonális rendszert a progonka módszerrel!

$$
\begin{bmatrix}
2 & -1 & 0 & 0 \\
-1 & 2 & -1 & 0 \\
0 & -1 & 2 & -1 \\
0 & 0 & -1 & 2
\end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \end{bmatrix}
=
\begin{bmatrix} 1 \\ 0 \\ 0 \\ 1 \end{bmatrix}
$$

Azonosítsuk az elemeket:
- $\alpha = [2, 2, 2, 2]$ (főátló)
- $\beta = [-1, -1, -1]$ (alsó mellékátló)
- $\gamma = [-1, -1, -1]$ (felső mellékátló)
- $b = [1, 0, 0, 1]$

In [ ]:
# Tridiagonális rendszer adatai
n = 4
alpha = np.array([2.0, 2.0, 2.0, 2.0])     # főátló (n elem)
beta  = np.array([-1.0, -1.0, -1.0])         # alsó mellékátló (n-1 elem)
gamma = np.array([-1.0, -1.0, -1.0])         # felső mellékátló (n-1 elem)
b     = np.array([1.0, 0.0, 0.0, 1.0])

# Teljes mátrix felépítése (csak ellenőrzéshez)
A_full = np.diag(alpha) + np.diag(beta, -1) + np.diag(gamma, 1)
print_matrix(A_full, "A =")
print(f"b = {b}")

In [ ]:
# === 1. lépés: előre (f_i, g_i számítása) ===
print("1. lépés: előre (f_i, g_i rekurzió)")
print("=" * 55)

f = np.zeros(n)
g = np.zeros(n)

# i = 1 (index 0)
f[0] = -gamma[0] / alpha[0]
g[0] = b[0] / alpha[0]
print(f"  f1 = -γ1/α1 = -({gamma[0]:.1f})/({alpha[0]:.1f}) = {f[0]:.4f}")
print(f"  g1 = b1/α1  = {b[0]:.1f}/{alpha[0]:.1f} = {g[0]:.4f}")
print()

# i = 2, ..., n-1 (index 1, ..., n-2)
for i in range(1, n - 1):
    nevező = alpha[i] + beta[i-1] * f[i-1]
    f[i] = -gamma[i] / nevező
    g[i] = (b[i] - beta[i-1] * g[i-1]) / nevező
    print(f"  i = {i+1}:")
    print(f"    nevező = α{i+1} + β{i}·f{i} = {alpha[i]:.1f} + ({beta[i-1]:.1f})·({f[i-1]:.4f}) = {nevező:.4f}")
    print(f"    f{i+1} = -γ{i+1}/nevező = -({gamma[i]:.1f})/({nevező:.4f}) = {f[i]:.4f}")
    print(f"    g{i+1} = (b{i+1} - β{i}·g{i})/nevező = ({b[i]:.1f} - ({beta[i-1]:.1f})·({g[i-1]:.4f}))/({nevező:.4f}) = {g[i]:.4f}")
    print()

# i = n (index n-1)
i = n - 1
nevező = alpha[i] + beta[i-1] * f[i-1]
g[i] = (b[i] - beta[i-1] * g[i-1]) / nevező
print(f"  i = {n}:")
print(f"    nevező = α{n} + β{n-1}·f{n-1} = {alpha[i]:.1f} + ({beta[i-1]:.1f})·({f[i-1]:.4f}) = {nevező:.4f}")
print(f"    g{n} = (b{n} - β{n-1}·g{n-1})/nevező = ({b[i]:.1f} - ({beta[i-1]:.1f})·({g[i-1]:.4f}))/({nevező:.4f}) = {g[i]:.4f}")

print(f"\n  f = {f}")
print(f"  g = {g}")

In [ ]:
# === 2. lépés: vissza (x_i = f_i * x_{i+1} + g_i) ===
print("2. lépés: visszahelyettesítés")
print("=" * 55)

x = np.zeros(n)
x[n-1] = g[n-1]
print(f"  x{n} = g{n} = {x[n-1]:.4f}")

for i in range(n - 2, -1, -1):
    x[i] = f[i] * x[i+1] + g[i]
    print(f"  x{i+1} = f{i+1}·x{i+2} + g{i+1} = ({f[i]:.4f})·({x[i+1]:.4f}) + ({g[i]:.4f}) = {x[i]:.4f}")

print(f"\nMegoldás: x = {x}")
print(f"Ellenőrzés: A @ x = {A_full @ x}")
print(f"A @ x = b? {np.allclose(A_full @ x, b)}")

In [ ]:
def progonka(alpha, beta, gamma, b):
    """Progonka módszer tridiagonális rendszerre.
    
    Paraméterek:
        alpha: főátló (n elem)
        beta:  alsó mellékátló (n-1 elem)
        gamma: felső mellékátló (n-1 elem)
        b:     jobboldal (n elem)
    
    Visszatérés:
        x: megoldásvektor (n elem)
    """
    n = len(alpha)
    f = np.zeros(n)
    g = np.zeros(n)
    
    # Előre
    f[0] = -gamma[0] / alpha[0]
    g[0] = b[0] / alpha[0]
    for i in range(1, n - 1):
        nev = alpha[i] + beta[i-1] * f[i-1]
        f[i] = -gamma[i] / nev
        g[i] = (b[i] - beta[i-1] * g[i-1]) / nev
    nev = alpha[n-1] + beta[n-2] * f[n-2]
    g[n-1] = (b[n-1] - beta[n-2] * g[n-2]) / nev
    
    # Vissza
    x = np.zeros(n)
    x[n-1] = g[n-1]
    for i in range(n - 2, -1, -1):
        x[i] = f[i] * x[i+1] + g[i]
    
    return x

# Ellenőrzés
x_prog = progonka(alpha, beta, gamma, b)
x_numpy = np.linalg.solve(A_full, b)

print(f"Progonka megoldás:  x = {x_prog}")
print(f"numpy megoldás:     x = {x_numpy}")
print(f"Egyeznek? {np.allclose(x_prog, x_numpy)}")

---
## 2. LDU-felbontás

### Definíció

Az $A \in \mathbb{R}^{n \times n}$ mátrix **LDU-felbontásának** nevezzük az $A = L \cdot D \cdot U$ szorzatot, ha:
- $L \in \mathcal{L}_1$: egységátlójú alsó háromszögmátrix
- $D$: diagonális mátrix
- $U \in \mathcal{U}_1$: egységátlójú felső háromszögmátrix

### Előállítás LU-felbontásból

Ha az $A = L\tilde{U}$ LU-felbontásban $L \in \mathcal{L}_1$ (jó), $\tilde{U} \in \mathcal{U}$ (nem egységátlójú):

$$D := \text{diag}(\tilde{u}_{11}, \ldots, \tilde{u}_{nn}), \qquad U := D^{-1} \tilde{U}$$

Azaz $\tilde{U}$ minden $i$. sorát leosztjuk $\tilde{u}_{ii}$-vel. Ekkor:

$$A = L\tilde{U} = L D \underbrace{(D^{-1}\tilde{U})}_{U} = LDU$$

### "Közvetlen" kiszámítás

Az $L$, $D$ és $U$ mátrixok elemeit a következő képletekkel számolhatjuk az LU-felbontás "közvetlen" képleteiből:

$$i < j \text{ (felső):} \quad u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik} \cdot d_{kk} \cdot u_{kj}$$

$$i = j \text{ (diag):} \quad d_{ii} = a_{ii} - \sum_{k=1}^{i-1} l_{ik} \cdot d_{kk} \cdot u_{ki}$$

$$i > j \text{ (alsó):} \quad l_{ij} = \frac{1}{d_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik} \cdot d_{kk} \cdot u_{kj} \right)$$

### Kidolgozott példa: LDU-felbontás LU-felbontásból

Készítsük el a következő mátrix LDU-felbontását!

$$
A = \begin{bmatrix}
2 & 0 & 3 \\
-4 & 5 & -2 \\
6 & -5 & 4
\end{bmatrix}
$$

In [ ]:
A_ldu = np.array([[2, 0, 3],
                  [-4, 5, -2],
                  [6, -5, 4]], dtype=float)

print_matrix(A_ldu, "A =")

# === 1. lépés: LU-felbontás GE-vel ===
print("=== 1. lépés: LU-felbontás GE-vel ===")
U_tilde = A_ldu.copy()

# 1. oszlop eliminációja
m21 = U_tilde[1, 0] / U_tilde[0, 0]
m31 = U_tilde[2, 0] / U_tilde[0, 0]
print(f"  m21 = {A_ldu[1,0]:.0f}/{A_ldu[0,0]:.0f} = {m21:.1f}")
print(f"  m31 = {A_ldu[2,0]:.0f}/{A_ldu[0,0]:.0f} = {m31:.1f}")
U_tilde[1] -= m21 * U_tilde[0]
U_tilde[2] -= m31 * U_tilde[0]
print_matrix(U_tilde, "  1. lépés után:")

# 2. oszlop eliminációja
m32 = U_tilde[2, 1] / U_tilde[1, 1]
print(f"  m32 = {U_tilde[2,1]:.0f}/{U_tilde[1,1]:.0f} = {m32:.1f}")
U_tilde[2] -= m32 * U_tilde[1]
print_matrix(U_tilde, "  Ũ =")

L = np.array([[1, 0, 0],
              [m21, 1, 0],
              [m31, m32, 1]])
print_matrix(L, "L =")
print(f"Ellenőrzés: L @ Ũ = A? {np.allclose(L @ U_tilde, A_ldu)}")

In [ ]:
# === 2. lépés: LDU előállítása ===
print("=== 2. lépés: LDU előállítása ===")

D = np.diag(np.diag(U_tilde))
print_matrix(D, f"D = diag({np.diag(U_tilde)}) =")

U = np.diag(1.0 / np.diag(U_tilde)) @ U_tilde
print("U = D^(-1) · Ũ:")
for i in range(3):
    print(f"  {i+1}. sor: [{', '.join(f'{U_tilde[i,j]:.0f}' for j in range(3))}] / {U_tilde[i,i]:.0f} = [{', '.join(f'{U[i,j]:.4f}' for j in range(3))}]")
print_matrix(U, "\nU =")

print(f"Ellenőrzés: L @ D @ U = A? {np.allclose(L @ D @ U, A_ldu)}")
print_matrix(L @ D @ U, "L @ D @ U =")

### Szimmetrikus mátrix LDU-felbontása: $A = LDL^\top$

**Tétel:** Ha $A$ szimmetrikus mátrix, akkor az LDU-felbontásában $U = L^\top$.

**Bizonyítás:** Az $A = LDU$ felbontás bal oldalát szorozzuk $L^{-1}$-zel, jobb oldalát $(L^{-1})^\top$-nal:

$$L^{-1} A (L^{-1})^\top = L^{-1} (LDU) (L^{-1})^\top = DU(L^{-1})^\top$$

A bal oldali mátrixról tudjuk, hogy szimmetrikus. A jobboldali felső háromszögmátrix, ami egyben szimmetrikus is, tehát diagonális. Ebből $U(L^{-1})^\top = I$, azaz $U = L^\top$.

**Következmény:** Szimmetrikus mátrix esetén az $LDL^\top$-felbontás:
- A teljes mátrix helyett elég $L$ és $D$ alsó háromszög részét tárolni
- A tárolás- és műveletigény kb. **felére** csökken ($\frac{1}{3}n^3 + \mathcal{O}(n^2)$)

### Az $LDL^\top$-felbontás "közvetlen" kiszámítása

$$i = j: \quad d_{ii} = a_{ii} - \sum_{k=1}^{i-1} l_{ik} \cdot d_{kk} \cdot l_{ik}$$

$$i > j: \quad l_{ij} = \frac{1}{d_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik} \cdot d_{kk} \cdot l_{jk} \right)$$

Ha jó sorrendben számolunk, mindig ismert az egész jobb oldal.

### Kidolgozott példa: szimmetrikus $LDL^\top$-felbontás

Készítsük el a következő szimmetrikus mátrix $LDL^\top$-felbontását a GE segítségével!

$$
A = \begin{bmatrix}
1 & 2 & 1 \\
2 & 8 & 6 \\
1 & 6 & 6
\end{bmatrix}
$$

In [ ]:
A_sym = np.array([[1, 2, 1],
                  [2, 8, 6],
                  [1, 6, 6]], dtype=float)

print_matrix(A_sym, "A =")
print(f"Szimmetrikus? {np.allclose(A_sym, A_sym.T)}")

# GE-vel, a hányadosokat az eliminált pozíciókon tároljuk
print("\n=== GE lépések (hányadosokat is tároljuk) ===")
W = A_sym.copy()

# 1. lépés
print("\n1. lépés:")
m21 = W[1, 0] / W[0, 0]
m31 = W[2, 0] / W[0, 0]
print(f"  m21 = {W[1,0]:.0f}/{W[0,0]:.0f} = {m21:.0f}")
print(f"  m31 = {W[2,0]:.0f}/{W[0,0]:.0f} = {m31:.0f}")
W[1, 1:] -= m21 * W[0, 1:]
W[2, 1:] -= m31 * W[0, 1:]
W[1, 0] = m21  # tároljuk a hányadost
W[2, 0] = m31
print_matrix(W, "  (hányadosokkal):")

# 2. lépés
print("2. lépés:")
m32 = W[2, 1] / W[1, 1]
print(f"  m32 = {W[2,1]:.0f}/{W[1,1]:.0f} = {m32:.0f}")
W[2, 2:] -= m32 * W[1, 2:]
W[2, 1] = m32
print_matrix(W, "  (hányadosokkal):")

# Leolvasás
L_sym = np.array([[1, 0, 0],
                  [W[1,0], 1, 0],
                  [W[2,0], W[2,1], 1]])
D_sym = np.diag([W[0,0], W[1,1], W[2,2]])

print_matrix(L_sym, "L =")
print_matrix(D_sym, "D =")
print_matrix(L_sym.T, "L^T =")

print(f"\nEllenőrzés: L @ D @ L^T = A? {np.allclose(L_sym @ D_sym @ L_sym.T, A_sym)}")
print_matrix(L_sym @ D_sym @ L_sym.T, "L @ D @ L^T =")

---
## 3. Cholesky-felbontás ($LL^\top$)

### Definíció

Az $A \in \mathbb{R}^{n \times n}$ szimmetrikus mátrix **Cholesky-felbontásának** (avagy $LL^\top$-felbontásának) nevezzük az $A = L \cdot L^\top$ szorzatot, ahol $L \in \mathbb{R}^{n \times n}$ alsó háromszögmátrix és $l_{ii} > 0$ ($i = 1, \ldots, n$).

### Létezés és egyértelműség

**Tétel:** Ha $A$ szimmetrikus és **pozitív definit** mátrix, akkor egyértelműen létezik Cholesky-felbontása.

**Bizonyítás vázlat (egyértelműség):** Indirekt. Tegyük fel, hogy $A = L_1 L_1^\top = L_2 L_2^\top$. Legyen $D_k = \text{diag}((L_k)_{ii})$. Ekkor:

$$(L_1 D_1^{-1}) \cdot (D_1 L_1^\top) = (L_2 D_2^{-1}) \cdot (D_2 L_2^\top)$$

Mindkét oldalon egy-egy LU-felbontást látunk. Az LU-felbontás egyértelműsége miatt $D_1 L_1^\top = D_2 L_2^\top$. A főátlóban $(L_1)_{ii}^2 = (L_2)_{ii}^2$, és a pozitivitás miatt $L_1 = L_2$. Ellentmondás.

**Bizonyítás vázlat (létezés):** Mivel $A$ pozitív definit, $\exists!\, A = L\tilde{U}$ LU-felbontás, ahol $\tilde{u}_{ii} > 0$ minden $i$-re. Legyen $D = \text{diag}(\sqrt{\tilde{u}_{11}}, \ldots, \sqrt{\tilde{u}_{nn}})$, így:

$$A = \underbrace{(\tilde{L}D)}_{B} \cdot \underbrace{(D^{-1}\tilde{U})}_{C} = B \cdot C$$

A szimmetria miatt $C^\top = B$, tehát $A = B \cdot B^\top = L \cdot L^\top$.

### Miért jó az $LL^\top$-felbontás?

Tegyük fel, hogy $A$ szimmetrikus, pozitív definit, és rendelkezésünkre áll az $A = LL^\top$ felbontás.

Ekkor $Ax = b$ megoldása:

1. Oldjuk meg az $Ly = b$ alsó háromszögű LER-t ($n^2 + \mathcal{O}(n)$)
2. Oldjuk meg az $L^\top x = y$ felső háromszögű LER-t ($n^2 + \mathcal{O}(n)$)

**Előnyök:**
- Csak $L$-et kell tárolni (nem $L$ és $U$-t külön)
- A felbontás műveletigénye: $\frac{1}{3}n^3 + \mathcal{O}(n^2)$ — **fele** az általános LU-felbontásnak
- Különösen előnyös, ha ugyanazzal az $A$-val sok jobboldalt kell megoldani

### Cholesky-felbontás előállítása: 3 módszer

#### 1. módszer: LU-felbontásból LDU-n keresztül

- $A = L\tilde{U}$ (LU-felbontás GE-vel)
- $D = \text{diag}(\sqrt{\tilde{u}_{11}}, \ldots, \sqrt{\tilde{u}_{nn}})$
- Cholesky: $L_{\text{Ch}} = \tilde{L} \cdot D$, azaz $\tilde{L}$ oszlopait a megfelelő $\sqrt{\tilde{u}_{ii}}$-vel szorozzuk

#### 2. módszer: "mechanikusan" a GE-ből

- $a_{11}$ helyére $\sqrt{a_{11}}$-et írunk
- Az 1. oszlopot végig elosztjuk $\sqrt{a_{11}}$-gyel
- Elvégezzük az eliminációt a jobb alsó részen
- Ismétlés a kisebb mátrixra...
- Az utolsó átlóelemből is gyököt vonunk

#### 3. módszer: "közvetlen" képletek (mátrixszorzás alapján)

$$i = j \text{ (átló):} \quad l_{jj} = \sqrt{a_{jj} - \sum_{k=1}^{j-1} l_{jk}^2}$$

$$i > j \text{ (alsó):} \quad l_{ij} = \frac{1}{l_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik} \cdot l_{jk} \right)$$

Ha jó sorrendben számolunk, mindig ismert az egész jobb oldal.

### Kidolgozott példa: Cholesky-felbontás mindhárom módszerrel

$$
A = \begin{bmatrix}
4 & 2 & 4 \\
2 & 10 & 5 \\
4 & 5 & 6
\end{bmatrix}
$$

Ellenőrizzük, hogy $A$ szimmetrikus és pozitív definit!

In [ ]:
A_ch = np.array([[4, 2, 4],
                 [2, 10, 5],
                 [4, 5, 6]], dtype=float)

print_matrix(A_ch, "A =")
print(f"Szimmetrikus? {np.allclose(A_ch, A_ch.T)}")

eigvals = np.linalg.eigvalsh(A_ch)
print(f"Sajátértékek: {eigvals}")
print(f"Pozitív definit? {np.all(eigvals > 0)}")

In [ ]:
# === 1. módszer: LU-felbontásból ===
print("=" * 55)
print("1. módszer: LU-felbontásból")
print("=" * 55)

# GE lépések
U_t = A_ch.copy()

# 1. lépés
m21 = U_t[1, 0] / U_t[0, 0]
m31 = U_t[2, 0] / U_t[0, 0]
print(f"  m21 = {U_t[1,0]:.0f}/{U_t[0,0]:.0f} = {m21:.4f}")
print(f"  m31 = {U_t[2,0]:.0f}/{U_t[0,0]:.0f} = {m31:.4f}")
U_t[1] -= m21 * U_t[0]
U_t[2] -= m31 * U_t[0]

# 2. lépés
m32 = U_t[2, 1] / U_t[1, 1]
print(f"  m32 = {U_t[2,1]:.4f}/{U_t[1,1]:.4f} = {m32:.4f}")
U_t[2] -= m32 * U_t[1]

L_tilde = np.array([[1, 0, 0],
                    [m21, 1, 0],
                    [m31, m32, 1]])

print_matrix(L_tilde, "\nL̃ =")
print_matrix(U_t, "Ũ =")

# D = diag(sqrt(ũ_ii))
d_vals = np.diag(U_t)
print(f"Ũ főátlója: {d_vals}")
sqrt_d = np.sqrt(d_vals)
print(f"sqrt: {sqrt_d}")
D_sqrt = np.diag(sqrt_d)

# L_ch = L_tilde * D_sqrt
L_ch1 = L_tilde @ D_sqrt
print_matrix(L_ch1, "\nL = L̃ · D =")

print(f"Ellenőrzés: L @ L^T = A? {np.allclose(L_ch1 @ L_ch1.T, A_ch)}")
print_matrix(L_ch1 @ L_ch1.T, "L @ L^T =")

In [ ]:
# === 2. módszer: "mechanikusan" a GE-ből ===
print("=" * 55)
print('2. módszer: "mechanikusan" a GE-ből')
print("=" * 55)

W = A_ch.copy()
print_matrix(W, "Kiindulás:")

# 1. lépés: a11 → sqrt(a11), 1. oszlop / sqrt(a11)
sqrt_a11 = np.sqrt(W[0, 0])
print(f"  sqrt(a11) = sqrt({W[0,0]:.0f}) = {sqrt_a11:.1f}")
W[:, 0] = W[:, 0] / sqrt_a11  # 1. oszlop leosztva
W[0, 0] = sqrt_a11             # a11 helyére sqrt
print(f"  1. oszlopot elosztjuk {sqrt_a11:.1f}-vel:")
print_matrix(W)

# Elimináció: jobb alsó 2x2 rész
print("  Elimináció az 1. sor segítségével (jobb alsó 2×2):")
W[1, 1:] -= W[1, 0] / W[0, 0] * W[0, 1:]
W[2, 1:] -= W[2, 0] / W[0, 0] * W[0, 1:]
print_matrix(W)

# 2. lépés: a22 → sqrt(a22), 2. oszlop / sqrt(a22)
sqrt_a22 = np.sqrt(W[1, 1])
print(f"  sqrt(a22) = sqrt({W[1,1]:.0f}) = {sqrt_a22:.1f}")
W[2, 1] = W[2, 1] / sqrt_a22
W[1, 1] = sqrt_a22
print_matrix(W)

# Elimináció: jobb alsó 1x1
print("  Elimináció (jobb alsó 1×1):")
W[2, 2] -= W[2, 1] / W[1, 1] * W[1, 2]
print_matrix(W)

# Utolsó átlóelem: gyökvonás
W[2, 2] = np.sqrt(W[2, 2])
print(f"  Utolsó átlóelemből gyököt vonunk: sqrt → {W[2,2]:.4f}")

# L leolvasása (alsó háromszög)
L_ch2 = np.tril(W)
print_matrix(L_ch2, "\nL =")
print(f"Ellenőrzés: L @ L^T = A? {np.allclose(L_ch2 @ L_ch2.T, A_ch)}")

In [ ]:
# === 3. módszer: "közvetlen" képletek ===
print("=" * 55)
print('3. módszer: "közvetlen" képletek')
print("=" * 55)

n = 3
L_ch3 = np.zeros((n, n))

# j = 1 (index 0)
L_ch3[0, 0] = np.sqrt(A_ch[0, 0])
print(f"  l11 = sqrt(a11) = sqrt({A_ch[0,0]:.0f}) = {L_ch3[0,0]:.4f}")

L_ch3[1, 0] = A_ch[1, 0] / L_ch3[0, 0]
print(f"  l21 = a21/l11 = {A_ch[1,0]:.0f}/{L_ch3[0,0]:.4f} = {L_ch3[1,0]:.4f}")

L_ch3[2, 0] = A_ch[2, 0] / L_ch3[0, 0]
print(f"  l31 = a31/l11 = {A_ch[2,0]:.0f}/{L_ch3[0,0]:.4f} = {L_ch3[2,0]:.4f}")

# j = 2 (index 1)
print()
val = A_ch[1, 1] - L_ch3[1, 0]**2
L_ch3[1, 1] = np.sqrt(val)
print(f"  l22 = sqrt(a22 - l21²) = sqrt({A_ch[1,1]:.0f} - {L_ch3[1,0]:.4f}²) = sqrt({val:.4f}) = {L_ch3[1,1]:.4f}")

val32 = A_ch[2, 1] - L_ch3[2, 0] * L_ch3[1, 0]
L_ch3[2, 1] = val32 / L_ch3[1, 1]
print(f"  l32 = (a32 - l31·l21)/l22 = ({A_ch[2,1]:.0f} - {L_ch3[2,0]:.4f}·{L_ch3[1,0]:.4f})/{L_ch3[1,1]:.4f} = {val32:.4f}/{L_ch3[1,1]:.4f} = {L_ch3[2,1]:.4f}")

# j = 3 (index 2)
print()
val33 = A_ch[2, 2] - L_ch3[2, 0]**2 - L_ch3[2, 1]**2
L_ch3[2, 2] = np.sqrt(val33)
print(f"  l33 = sqrt(a33 - l31² - l32²) = sqrt({A_ch[2,2]:.0f} - {L_ch3[2,0]:.4f}² - {L_ch3[2,1]:.4f}²) = sqrt({val33:.4f}) = {L_ch3[2,2]:.4f}")

print_matrix(L_ch3, "\nL =")
print(f"Ellenőrzés: L @ L^T = A? {np.allclose(L_ch3 @ L_ch3.T, A_ch)}")
print_matrix(L_ch3 @ L_ch3.T, "L @ L^T =")

In [ ]:
# Összehasonlítás scipy.linalg.cholesky-val
from scipy.linalg import cholesky

L_scipy = cholesky(A_ch, lower=True)
print_matrix(L_scipy, "scipy cholesky L =")
print(f"A mi L-ünk egyezik? {np.allclose(L_ch3, L_scipy)}")

### A Cholesky-felbontás műveletigénye

A szorzások és osztások száma:

$$\frac{1}{3}n^3 + \mathcal{O}(n^2)$$

valamint $n$ darab négyzetgyökvonás is szükséges.

**Bizonyítás:** A képletekből:
- Rögzített $j$-re: $l_{jj}$-hez $2(j-1)$ szorzás és összeadás kell.
- Rögzített $i, j$-re: $l_{ij}$-hez 1 osztás, $(j-1)$ szorzás és $(j-1)$ összeadás kell. Összesen $2j - 1$ művelet.

$$\sum_{j=1}^n 2(j-1) + \sum_{i=2}^n \sum_{j=1}^{i-1} (2j-1) = \frac{1}{3}n^3 + \mathcal{O}(n^2)$$

### LER megoldása Cholesky-felbontással

Oldjuk meg az $Ax = b$ rendszert a fenti $A$ mátrixszal és $b = [4, 13, 11]^\top$ jobboldallal!

In [ ]:
b_ch = np.array([4, 13, 11], dtype=float)

print("Ax = b megoldása Cholesky-felbontással")
print("A = L·L^T, L ismert.\n")

# 1. Ly = b (előre helyettesítés)
print("1. fázis: Ly = b (előre helyettesítés)")
print("=" * 50)
L_ch = L_ch3  # az előző példából
y = np.zeros(3)

y[0] = b_ch[0] / L_ch[0, 0]
print(f"  y1 = b1/l11 = {b_ch[0]:.0f}/{L_ch[0,0]:.4f} = {y[0]:.4f}")

y[1] = (b_ch[1] - L_ch[1, 0] * y[0]) / L_ch[1, 1]
print(f"  y2 = (b2 - l21·y1)/l22 = ({b_ch[1]:.0f} - {L_ch[1,0]:.4f}·{y[0]:.4f})/{L_ch[1,1]:.4f} = {y[1]:.4f}")

y[2] = (b_ch[2] - L_ch[2, 0] * y[0] - L_ch[2, 1] * y[1]) / L_ch[2, 2]
print(f"  y3 = (b3 - l31·y1 - l32·y2)/l33 = ({b_ch[2]:.0f} - {L_ch[2,0]:.4f}·{y[0]:.4f} - {L_ch[2,1]:.4f}·{y[1]:.4f})/{L_ch[2,2]:.4f} = {y[2]:.4f}")

print(f"\n  y = {y}")

# 2. L^T x = y (visszahelyettesítés)
print(f"\n2. fázis: L^T x = y (visszahelyettesítés)")
print("=" * 50)
LT = L_ch.T
x_ch = np.zeros(3)

x_ch[2] = y[2] / LT[2, 2]
print(f"  x3 = y3/l33 = {y[2]:.4f}/{LT[2,2]:.4f} = {x_ch[2]:.4f}")

x_ch[1] = (y[1] - LT[1, 2] * x_ch[2]) / LT[1, 1]
print(f"  x2 = (y2 - l32·x3)/l22 = ({y[1]:.4f} - {LT[1,2]:.4f}·{x_ch[2]:.4f})/{LT[1,1]:.4f} = {x_ch[1]:.4f}")

x_ch[0] = (y[0] - LT[0, 1] * x_ch[1] - LT[0, 2] * x_ch[2]) / LT[0, 0]
print(f"  x1 = (y1 - l21·x2 - l31·x3)/l11 = ({y[0]:.4f} - {LT[0,1]:.4f}·{x_ch[1]:.4f} - {LT[0,2]:.4f}·{x_ch[2]:.4f})/{LT[0,0]:.4f} = {x_ch[0]:.4f}")

print(f"\nMegoldás: x = {x_ch}")
print(f"Ellenőrzés: A @ x = {A_ch @ x_ch}")
print(f"A @ x = b? {np.allclose(A_ch @ x_ch, b_ch)}")

---
## 4. Önálló feladatok

Az alábbi feladatokat a fenti példák mintájára oldják meg! Először próbálják meg **kézzel** (papír-ceruza), majd ellenőrizzék Pythonnal!

### 1. feladat: Progonka módszer

Oldjuk meg a progonka módszerrel az alábbi 5×5 tridiagonális rendszert! Írjuk ki lépésenként az $f_i$, $g_i$ értékeket, majd a visszahelyettesítés eredményét!

$$
\begin{bmatrix}
3 & -1 & 0 & 0 & 0 \\
-1 & 3 & -1 & 0 & 0 \\
0 & -1 & 3 & -1 & 0 \\
0 & 0 & -1 & 3 & -1 \\
0 & 0 & 0 & -1 & 3
\end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ x_5 \end{bmatrix}
=
\begin{bmatrix} 2 \\ 1 \\ 0 \\ 1 \\ 2 \end{bmatrix}
$$

In [ ]:
# 1. feladat -- megoldás helye
# Tipp: használd a progonka függvényt, vagy írd ki kézzel a lépéseket!

alpha_f1 = np.array([3.0, 3.0, 3.0, 3.0, 3.0])
beta_f1  = np.array([-1.0, -1.0, -1.0, -1.0])
gamma_f1 = np.array([-1.0, -1.0, -1.0, -1.0])
b_f1     = np.array([2.0, 1.0, 0.0, 1.0, 2.0])

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 1. feladat -- ellenőrzés
A_f1 = np.diag(alpha_f1) + np.diag(beta_f1, -1) + np.diag(gamma_f1, 1)
x_f1 = np.linalg.solve(A_f1, b_f1)
print(f"numpy megoldás: x = {x_f1}")

### 2. feladat: LDU-felbontás

Készítsük el a következő mátrix LDU-felbontását! Írjuk ki az $L$, $D$, $U$ mátrixokat!

$$
A = \begin{bmatrix}
3 & 1 & -1 \\
6 & 4 & 2 \\
-3 & 2 & 1
\end{bmatrix}
$$

In [ ]:
# 2. feladat -- megoldás helye
A_f2 = np.array([[3, 1, -1],
                 [6, 4, 2],
                 [-3, 2, 1]], dtype=float)

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 2. feladat -- ellenőrzés
P_f2, L_f2, U_f2 = lu(A_f2)
if np.allclose(P_f2, np.eye(3)):
    D_diag = np.diag(U_f2)
    D_f2 = np.diag(D_diag)
    U_f2_norm = np.diag(1.0/D_diag) @ U_f2
    print_matrix(L_f2, "L =")
    print_matrix(D_f2, "D =")
    print_matrix(U_f2_norm, "U =")
    print(f"L @ D @ U = A? {np.allclose(L_f2 @ D_f2 @ U_f2_norm, A_f2)}")

### 3. feladat: $LDL^\top$-felbontás

Készítsük el az alábbi szimmetrikus mátrix $LDL^\top$-felbontását!

$$
A = \begin{bmatrix}
4 & 2 & -2 \\
2 & 5 & 1 \\
-2 & 1 & 6
\end{bmatrix}
$$

Ellenőrizzük, hogy $U = L^\top$!

In [ ]:
# 3. feladat -- megoldás helye
A_f3 = np.array([[4, 2, -2],
                 [2, 5, 1],
                 [-2, 1, 6]], dtype=float)

# IDE ÍRD A MEGOLDÁST!


### 4. feladat: Cholesky-felbontás közvetlen képletekkel

Készítsük el a következő szimmetrikus, pozitív definit mátrix Cholesky-felbontását a **közvetlen képletekkel**!

$$
A = \begin{bmatrix}
9 & 6 & 3 \\
6 & 20 & 10 \\
3 & 10 & 14
\end{bmatrix}
$$

Számítsuk ki lépésenként: $l_{11}, l_{21}, l_{31}, l_{22}, l_{32}, l_{33}$.

In [ ]:
# 4. feladat -- megoldás helye
A_f4 = np.array([[9, 6, 3],
                 [6, 20, 10],
                 [3, 10, 14]], dtype=float)

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 4. feladat -- ellenőrzés
L_f4 = cholesky(A_f4, lower=True)
print_matrix(L_f4, "scipy Cholesky L =")
print(f"L @ L^T = A? {np.allclose(L_f4 @ L_f4.T, A_f4)}")

### 5. feladat: Cholesky létezés ellenőrzése

Az alábbi mátrixok közül melyikre létezik Cholesky-felbontás? Indokoljuk!

$$
A_1 = \begin{bmatrix} 4 & 2 \\ 2 & 1 \end{bmatrix}, \quad
A_2 = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}, \quad
A_3 = \begin{bmatrix} 5 & -1 \\ -1 & 5 \end{bmatrix}
$$

Ellenőrizzük a feltételeket: **szimmetria** és **pozitív definitség** (sajátértékek > 0).

In [ ]:
# 5. feladat -- megoldás helye
A1_f5 = np.array([[4, 2], [2, 1]], dtype=float)
A2_f5 = np.array([[1, 2], [3, 4]], dtype=float)
A3_f5 = np.array([[5, -1], [-1, 5]], dtype=float)

for name, M in [("A1", A1_f5), ("A2", A2_f5), ("A3", A3_f5)]:
    szimm = np.allclose(M, M.T)
    eigv = np.linalg.eigvalsh(M) if szimm else np.linalg.eigvals(M)
    pozdef = szimm and np.all(eigv > 0)
    print(f"{name}: szimmetrikus={szimm}, sajátértékek={eigv}, poz.def.={pozdef}")
    if pozdef:
        L_tmp = cholesky(M, lower=True)
        print(f"  → Cholesky létezik! L = {L_tmp}")
    else:
        print(f"  → Cholesky NEM létezik!")
    print()

### 6. feladat: Progonka vs. általános GE futásidő

Hasonlítsuk össze a progonka módszer és az `np.linalg.solve` futásidejét különböző $n$ értékekre! Készítsünk grafikont!

*Tipp:* Használjunk tridiagonális rendszert $\alpha_i = 2$, $\beta_i = \gamma_i = -1$ főátlóval, és véletlenszerű jobboldallal.

In [ ]:
# 6. feladat -- megoldás helye
import matplotlib.pyplot as plt

n_values = [100, 500, 1000, 2000, 5000]
time_progonka = []
time_numpy = []

for n in n_values:
    alpha_t = 2.0 * np.ones(n)
    beta_t = -1.0 * np.ones(n - 1)
    gamma_t = -1.0 * np.ones(n - 1)
    b_t = np.random.randn(n)
    A_t = np.diag(alpha_t) + np.diag(beta_t, -1) + np.diag(gamma_t, 1)
    
    # Progonka
    t0 = time.perf_counter()
    for _ in range(10):
        x_p = progonka(alpha_t, beta_t, gamma_t, b_t)
    t_p = (time.perf_counter() - t0) / 10
    time_progonka.append(t_p)
    
    # numpy
    t0 = time.perf_counter()
    for _ in range(10):
        x_n = np.linalg.solve(A_t, b_t)
    t_n = (time.perf_counter() - t0) / 10
    time_numpy.append(t_n)
    
    print(f"n={n:5d}: progonka={t_p:.6f}s, numpy={t_n:.6f}s, gyorsítás={t_n/t_p:.1f}x")

plt.figure(figsize=(8, 5))
plt.plot(n_values, time_progonka, 'o-', label='Progonka O(n)')
plt.plot(n_values, time_numpy, 's-', label='numpy.linalg.solve O(n³)')
plt.xlabel('n (mátrix mérete)')
plt.ylabel('Futásidő (s)')
plt.title('Progonka vs. általános megoldó')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.xscale('log')
plt.tight_layout()
plt.show()

---
## 5. Debuggolási feladatok

Az alábbi kódokban **szándékos hibák** vannak. Keressük meg és javítsuk ki őket!

### Debug 1: Hibás progonka implementáció

Az alábbi progonka implementáció **hibás eredményt** ad. Keressük meg a hibát!

*Tipp:* Teszteld a fenti 4×4-es példával, és hasonlítsd össze a helyes megoldással! A hiba az indexelésben van.

In [ ]:
def progonka_hibas(alpha, beta, gamma, b):
    """HIBÁS progonka implementáció -- keresd meg a hibát!"""
    n = len(alpha)
    f = np.zeros(n)
    g = np.zeros(n)
    
    # Előre
    f[0] = -gamma[0] / alpha[0]
    g[0] = b[0] / alpha[0]
    for i in range(1, n - 1):
        nev = alpha[i] + beta[i] * f[i-1]       # <-- VIZSGÁLD MEG EZT A SORT!
        f[i] = -gamma[i] / nev
        g[i] = (b[i] - beta[i] * g[i-1]) / nev  # <-- ÉS EZT IS!
    nev = alpha[n-1] + beta[n-2] * f[n-2]
    g[n-1] = (b[n-1] - beta[n-2] * g[n-2]) / nev
    
    # Vissza
    x = np.zeros(n)
    x[n-1] = g[n-1]
    for i in range(n - 2, -1, -1):
        x[i] = f[i] * x[i+1] + g[i]
    
    return x

# Tesztelés
alpha_t = np.array([2.0, 2.0, 2.0, 2.0])
beta_t  = np.array([-1.0, -1.0, -1.0])
gamma_t = np.array([-1.0, -1.0, -1.0])
b_t     = np.array([1.0, 0.0, 0.0, 1.0])

x_hibas = progonka_hibas(alpha_t, beta_t, gamma_t, b_t)
A_t = np.diag(alpha_t) + np.diag(beta_t, -1) + np.diag(gamma_t, 1)
x_helyes = np.linalg.solve(A_t, b_t)

print(f"Hibás megoldás:  x = {x_hibas}")
print(f"Helyes megoldás: x = {x_helyes}")
print(f"Egyeznek? {np.allclose(x_hibas, x_helyes)}")
print(f"\nHiba: A @ x_hibás = {A_t @ x_hibas}  (kellene: {b_t})")
print("\n>>> Keresd meg és javítsd ki a hibát a progonka_hibas függvényben!")

### Debug 2: Hibás Cholesky implementáció

Az alábbi "közvetlen" Cholesky-implementáció hibás. Két különböző problémát is tartalmaz.

*Tipp 1:* Az egyik hiba a képletben van (figyelj az indexekre!).  
*Tipp 2:* A másik hiba: mi történik, ha nem pozitív definit mátrixra hívjuk meg?

In [ ]:
def cholesky_hibas(A):
    """HIBÁS Cholesky-felbontás -- keresd meg a hibákat!"""
    n = A.shape[0]
    L = np.zeros((n, n))
    
    for j in range(n):
        # Átlóelem
        s = 0.0
        for k in range(j):
            s += L[j, k] ** 2
        L[j, j] = np.sqrt(A[j, j] - s)  # Mi van, ha A[j,j] - s < 0?
        
        # Átló alatti elemek
        for i in range(j + 1, n):
            s = 0.0
            for k in range(j):
                s += L[i, k] * L[i, k]   # <-- VIZSGÁLD MEG EZT A SORT!
            L[i, j] = (A[i, j] - s) / L[j, j]
    
    return L

# Tesztelés poz. def. mátrixra
A_test = np.array([[4, 2, 4],
                   [2, 10, 5],
                   [4, 5, 6]], dtype=float)

L_hibas = cholesky_hibas(A_test)
L_helyes = cholesky(A_test, lower=True)

print_matrix(L_hibas, "Hibás L =")
print_matrix(L_helyes, "Helyes L =")
print(f"Egyeznek? {np.allclose(L_hibas, L_helyes)}")
print(f"L_hibás @ L_hibás^T = A? {np.allclose(L_hibas @ L_hibas.T, A_test)}")

print("\n>>> Keresd meg a két hibát a cholesky_hibas függvényben!")
print(">>> Tipp: mi történne, ha nem poz. def. mátrixra hívnád meg?")

### Debug 3: Hibás LDU-felbontás

Az alábbi kód LDU-felbontást állít elő, de a végeredmény **nem egyezik** az eredeti mátrixszal. Hol a hiba?

*Tipp:* A GE lépések helyesek, a hiba az $U$ mátrix előállításában van.

In [ ]:
def ldu_hibas(A):
    """HIBÁS LDU-felbontás -- keresd meg a hibát!"""
    n = A.shape[0]
    U_tilde = A.copy().astype(float)
    L = np.eye(n)
    
    # GE
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = U_tilde[i, k] / U_tilde[k, k]
            U_tilde[i, k:] -= m * U_tilde[k, k:]
            L[i, k] = m
    
    # D és U előállítása
    d = np.diag(U_tilde)
    D = np.diag(d)
    U = U_tilde.copy()
    for i in range(n):
        U[i, :] = U_tilde[i, :] / d[i]   # <-- VIZSGÁLD MEG EZT A SORT!
    
    return L, D, U

# Tesztelés
A_test2 = np.array([[2, 0, 3],
                    [-4, 5, -2],
                    [6, -5, 4]], dtype=float)

L_h, D_h, U_h = ldu_hibas(A_test2)

print_matrix(L_h, "L =")
print_matrix(D_h, "D =")
print_matrix(U_h, "U =")

result = L_h @ D_h @ U_h
print_matrix(result, "L @ D @ U =")
print_matrix(A_test2, "A =")
print(f"L @ D @ U = A? {np.allclose(result, A_test2)}")

print("\n>>> Keresd meg a hibát! Miért nem egyezik L @ D @ U az A-val?")
print(">>> Tipp: nézd meg az U mátrix átlóját -- mi kellene ott legyen?")

---
## Összefoglaló

| Téma | Lényeg |
|------|--------|
| **Progonka módszer** | Tridiagonális rendszer hatékony megoldása, $8n + \mathcal{O}(1)$ művelet |
| **LDU-felbontás** | $A = LDU$: LU-felbontásból $U$ sorait leosztjuk az átlóelemekkel |
| **$LDL^\top$-felbontás** | Szimmetrikus esetben $U = L^\top$, fele annyi tárolás/művelet |
| **Cholesky ($LL^\top$)** | Szimm. + poz. def. mátrixra, egyértelmű, $\frac{1}{3}n^3 + \mathcal{O}(n^2)$ + $n$ gyök |
| **Cholesky előállítás** | 3 módszer: (1) LU→LDU→LL$^\top$, (2) mechanikus GE, (3) közvetlen képletek |
| **Cholesky előny** | Fele annyi tárolás és művelet, mint az LU-felbontás |